<a href="https://colab.research.google.com/github/Datdeptrai912005/DemoGit/blob/main/MLLM_mBERT_Trend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install transformers datasets evaluate accelerate scikit-learn

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
def preprocess_function(examples):
    texts = []
    labels = []
    for conv in examples["messages"]:
        user_text = next((msg["content"] for msg in conv if msg["role"] == "user"), "")
        label_text = next((msg["content"] for msg in conv if msg["role"] == "assistant"), "0")

        texts.append(user_text)
        labels.append(int(label_text))

    result = tokenizer(texts, padding="max_length", truncation=True, max_length=512)
    result["labels"] = labels
    return result

print("Đang tải dữ liệu từ Drive...")
train_dataset = load_dataset("json", data_files="/content/drive/MyDrive/train_data.jsonl", split="train")
test_dataset = load_dataset("json", data_files="/content/drive/MyDrive/test_data.jsonl", split="train")

tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=["messages"])
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=["messages"])

print(f"✅ Đã nạp {len(tokenized_train)} mẫu Train và {len(tokenized_test)} mẫu Test cho mBERT.")

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

# 1. Định nghĩa cách chấm điểm (Accuracy và F1-Score)
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"accuracy": acc, "f1": f1}

# 2. Cấu hình thông số học tập
training_args = TrainingArguments(
    output_dir="./mbert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16, # mBERT nhẹ nên có thể nhồi 16 câu cùng lúc
    per_device_eval_batch_size=16,
    num_train_epochs=3, # mBERT thường cần học 3 vòng mới ngấm
    weight_decay=0.01,
    eval_strategy="epoch", # Chấm điểm sau mỗi vòng học
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True, # Tăng tốc độ train trên GPU T4
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer, # <-- TÔI ĐÃ SỬA DÒNG NÀY CHO BẠN
    compute_metrics=compute_metrics,
)
print("Bắt đầu huấn luyện mBERT...")
trainer.train()

print("\n" + "="*40)
print("🏆 KẾT QUẢ CUỐI CÙNG CỦA mBERT TRÊN TẬP TEST 🏆")
print("="*40)
final_results = trainer.evaluate(tokenized_test)
print(f"Độ chính xác (Accuracy): {final_results['eval_accuracy'] * 100:.2f}%")
print(f"Điểm F1-Score:           {final_results['eval_f1'] * 100:.2f}%")
print("="*40)

trainer.save_model("/content/drive/MyDrive/mBERT_FakeNews_Model")
print("Đã lưu mô hình mBERT lên Drive.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

print("Đang thu thập đáp án của mBERT...")
predictions = trainer.predict(tokenized_test)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tin thật (0)', 'Tin giả (1)'],
            yticklabels=['Tin thật (0)', 'Tin giả (1)'])

plt.title('Sơ đồ Phân tích lỗi mBERT', fontsize=15, fontweight='bold')
plt.xlabel('AI Dự Đoán', fontsize=12)
plt.ylabel('Đấp Án Thực Tế', fontsize=12)
plt.show()